In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DWH-to-ClickHouse-Marts") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.4,com.clickhouse:clickhouse-jdbc:0.4.6") \
    .getOrCreate()

pg_url = "jdbc:postgresql://postgres:5432/petshop_db"
pg_props = {"user": "admin", "password": "4321", "driver": "org.postgresql.Driver"}

ch_url = "jdbc:ch://clickhouse:8123/petshop_reports"
ch_props = {
    "user": "admin", 
    "password": "4321", 
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "host": "clickhouse",
    "port": "8123",
    "compress": "0",      
    "decompress": "0",
    "createTableOptions": "ENGINE = MergeTree() ORDER BY tuple()"
}

print("Spark готов. Соединение с хостами 'postgres' и 'clickhouse' настроено.")

Spark готов. Соединение с хостами 'postgres' и 'clickhouse' настроено.


In [2]:
print("Импорт таблиц из PostgreSQL...")

spark.read.jdbc(url=pg_url, table="star_fact_sales", properties=pg_props).createOrReplaceTempView("fact_sales")
spark.read.jdbc(url=pg_url, table="star_dim_products", properties=pg_props).createOrReplaceTempView("dim_products")
spark.read.jdbc(url=pg_url, table="star_dim_customers", properties=pg_props).createOrReplaceTempView("dim_customers")
spark.read.jdbc(url=pg_url, table="star_dim_stores", properties=pg_props).createOrReplaceTempView("dim_stores")
spark.read.jdbc(url=pg_url, table="star_dim_suppliers", properties=pg_props).createOrReplaceTempView("dim_suppliers")

print("Данные успешно загружены.")

Импорт таблиц из PostgreSQL...
Данные успешно загружены.


In [3]:
print("Генерация витрины: Продажи по продуктам...")
mart_products = spark.sql("""
    SELECT 
        p.p_name AS product_name, 
        p.p_category AS category, 
        SUM(f.total_price) AS total_revenue, 
        SUM(f.quantity) AS total_sold, 
        AVG(p.p_rating) AS avg_rating, 
        MAX(p.p_reviews_cnt) AS total_reviews
    FROM fact_sales f 
    JOIN dim_products p ON f.product_id = p.product_id
    GROUP BY p.p_name, p.p_category 
    ORDER BY total_sold DESC 
    LIMIT 10
""")
mart_products.write.jdbc(url=ch_url, table="mart_product_sales", mode="overwrite", properties=ch_props)
print("Готово.")

Генерация витрины: Продажи по продуктам...
Готово.


In [4]:
print("Генерация витрины: Анализ клиентов...")
mart_customers = spark.sql("""
    SELECT 
        c.c_first_name || ' ' || c.c_last_name AS customer_full_name, c.c_country AS country, 
        SUM(f.total_price) AS total_spent, AVG(f.total_price) AS avg_receipt
    FROM fact_sales f 
    JOIN dim_customers c ON f.customer_id = c.customer_id
    GROUP BY customer_full_name, country ORDER BY total_spent DESC LIMIT 10
""")
mart_customers.write.jdbc(url=ch_url, table="mart_customer_behavior", mode="overwrite", properties=ch_props)
print("Готово.")

Генерация витрины: Анализ клиентов...
Готово.


In [5]:
print("Генерация витрины: Тренды по времени...")
mart_time = spark.sql("""
    SELECT 
        YEAR(f.sale_date) AS sale_year,
        SUBSTRING(CAST(f.sale_date AS STRING), 1, 7) AS sale_month, 
        SUM(f.total_price) AS monthly_revenue, 
        AVG(f.total_price) AS avg_order_size
    FROM fact_sales f 
    GROUP BY sale_year, sale_month 
    ORDER BY sale_month
""")
mart_time.write.jdbc(url=ch_url, table="mart_time_trends", mode="overwrite", properties=ch_props)
print("Готово.")

Генерация витрины: Тренды по времени...
Готово.


In [6]:
print("Генерация витрины: Продажи по магазинам...")
mart_stores = spark.sql("""
    SELECT 
        s.shop_name, s.shop_city AS city, s.shop_country AS country, 
        SUM(f.total_price) AS store_revenue, AVG(f.total_price) AS avg_receipt
    FROM fact_sales f 
    JOIN dim_stores s ON f.store_id = s.store_id
    GROUP BY s.shop_name, city, country ORDER BY store_revenue DESC LIMIT 5
""")
mart_stores.write.jdbc(url=ch_url, table="mart_store_performance", mode="overwrite", properties=ch_props)
print("Готово.")

Генерация витрины: Продажи по магазинам...
Готово.


In [7]:
print("Генерация витрины: Анализ поставщиков...")
mart_suppliers = spark.sql("""
    SELECT 
        sup.vendor_name AS supplier_name, sup.vendor_country AS country, 
        SUM(f.total_price) AS total_supplier_revenue, AVG(p.p_price) AS avg_item_price
    FROM fact_sales f 
    JOIN dim_suppliers sup ON f.supplier_id = sup.supplier_id 
    JOIN dim_products p ON f.product_id = p.product_id
    GROUP BY supplier_name, country ORDER BY total_supplier_revenue DESC LIMIT 5
""")
mart_suppliers.write.jdbc(url=ch_url, table="mart_supplier_performance", mode="overwrite", properties=ch_props)
print("Готово.")

Генерация витрины: Анализ поставщиков...
Готово.


In [8]:
print("Генерация витрины: Качество товаров...")
mart_quality = spark.sql("""
    SELECT 
        p.p_name AS product_title, 
        AVG(p.p_rating) AS product_rating, 
        MAX(p.p_reviews_cnt) AS reviews_count, 
        SUM(f.quantity) AS sales_volume
    FROM fact_sales f 
    JOIN dim_products p ON f.product_id = p.product_id
    GROUP BY p.p_name 
    ORDER BY product_rating DESC
""")
mart_quality.write.jdbc(url=ch_url, table="mart_product_quality", mode="overwrite", properties=ch_props)
print("Готово.")

Генерация витрины: Качество товаров...
Готово.


In [9]:
print("Все отчеты выгружены в ClickHouse.")
spark.stop()

Все отчеты выгружены в ClickHouse.
